# 7.3 访存性能

上一节的 Roofline 把带宽 B 当成一个数；本节看它的两个真实陷阱：**多级带宽账单**与 **UB bank 冲突**（实测带宽远低于标称值的常见元凶）。

## 1. 多级存储的带宽账单

数据通路 GM → L2 → UB → (L0/RF)，每级的带宽和容量差数量级。任何一级成为短板，后面的级再快也没用。FA 各步的开销归属：

| 步骤 | 走的通路 | 优化抓手 |
|--|--|--|
| K/V 搬入（MTE2） | GM/L2 → UB | 量化（第 6 章）、预取（preload） |
| 搬入 Cube（MTE1） | UB → L0 | 基本块大小（上一节公式） |
| 结果搬出（MTE3/FixPipe） | L0/UB → GM | 输出布局、写合并 |

## 2. UB 的 bank 组织

UB（Unified Buffer）不是一块均匀内存，而是按 **bank group → bank → block（32B）** 三级组织：

| 芯片 | 组织形式 | 总量 | 交织方式 |
|--|--|--|--|
| David | 8 个 bank group × 2 bank × 4 block | 256KB | **低位交织**：bank0 与 bank1 地址相邻 256B，两个 bank 的 block 交替排布 |
| 1971 | 16 个 bank group × 3 bank × 128 block | 192KB | **高位交织**：每个 bank 连续存放 64KB，排完一个 bank 再排下一个 |

bank 组织形式随芯片代际变化，但 **block 固定 32B**。同一 bank 内部是顺序访问的——硬件只保证「不同 bank 并发、同一 bank 串行」。

## 3. Bank 冲突：定义、类型与判定

**读写同一个 bank 中的不同数据地址时，原本可以并发执行的读写退化成顺序执行**，就是 bank 冲突。VEC 读写、MTE 读写都可能产生。以 David（256 取模）为例：

| 类型 | 判定条件 |
|--|--|
| 读读 / 写写冲突 | 同时读/写的地址所在 block 的**起始地址对 256 取模余数相等**（1971 为 512） |
| 读写冲突 | 取模余数相等，**且起始地址之差可被 512 整除**（1971：不大于 65024） |

仲裁优先级：**VEC > Scalar > MTE**，每个 pipe 内 **Write > Read**——冲突时低优先级方被阻塞。

**两种判定方法**：

1. **定义判断**：拿两条并发指令的操作地址，按上表手工验算；
2. **dump log 分析法**：打开 `core0.vector0.ub.dump`，搜索 **`BGBK_CFLT`** 和 **`CFLT_BLOCKED`** 关键字——命中即发生冲突，按冲突记录里的两条指令 PC 指针回查 instr_log，定位到具体指令。

![UB Bank](images/ub_bank.svg)

## 4. 解冲突三板斧

1. **控制指令宽度**：单条 VEC 指令的操作不超过 256B；
2. **读写 pingpong**：计算输入数据与输出数据分别存放在 ping/pong 缓冲——直接消灭读写冲突（与第 5 章 buffer 乒乓是同一件事的两种动机）；
3. **block 均匀分布**：读/写操作的 8 个 block 尽量分布在 8 个不同的 bank group（写操作只有一个操作数，尤其要把写分散）。

> FA 里的直接应用：V1 读 score 矩阵 + V2 写 output 累加，输入输出在 UB 上的摆放如果踩到同一 bank 的同余地址，Vector 实测带宽直接打折——这是「kernel 写对了但性能上不去」的高频原因。

## 小测验

1. 写出 David 与 1971 的 bank 组织形式与交织差异。block 大小是多少？
2. 读读/写写冲突的判定条件是什么（David）？读写冲突在此基础上多了什么条件？
3. dump log 里搜哪两个关键字定位 bank 冲突？拿到冲突记录后怎么找到具体指令？
4. 解冲突三板斧分别消灭哪类冲突？「读写 pingpong」与第 5 章的乒乓缓冲是什么关系？

In [ ]:
# 动手：David 规则的 bank 冲突判定器
# David：读写/写写 = 起始地址 mod 256 相等；读写 = mod 256 相等 且 地址差可被 512 整除
def block_start(addr):
    return addr // 32 * 32          # block 32B 对齐

def conflict(addrA, addrB, kind):
    # kind: 'rr' 读读 / 'ww' 写写 / 'rw' 读写
    m = (block_start(addrA) % 256) == (block_start(addrB) % 256)
    if kind in ('rr', 'ww'):
        return m
    return m and (addrA - addrB) % 512 == 0

cases = [
    (0x0000, 0x0100, 'rr'),   # 0 与 256
    (0x0000, 0x0200, 'rr'),   # 0 与 512
    (0x0000, 0x0200, 'rw'),
    (0x0020, 0x0120, 'ww'),
    (0x0000, 0x0040, 'rr'),
]
for a, b, k in cases:
    print(f'{k.upper()}  0x{a:04X} vs 0x{b:04X} -> {"冲突!" if conflict(a, b, k) else "无冲突"}')

print()
print('观察：0 与 256 —— mod 256 相等 => 读读/写写冲突，但差 256 不能被 512 整除 => 不是读写冲突；')
print('     0 与 512 —— mod 相等且差 512 整除 => 读读/写写/读写三种都冲突；')
print('     0 与 64 —— mod 256 不等（错开 bank）=> 永不冲突。')